In [1]:
import os
from pathlib import Path

import pandas as pd
from googleapiclient.discovery import build


def _load_env_file(path: str = ".env") -> None:
    p = Path(path)
    if not p.exists():
        return

    for raw_line in p.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, value)


_load_env_file()


In [2]:
API_KEY = os.getenv("YOUTUBE_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "Missing YOUTUBE_API_KEY. Create a .env file (see .env.example) or set it in your environment."
    )

In [3]:
youtube = build("youtube", "v3", developerKey=API_KEY)


In [4]:
def get_channel_id(channel_name):
    request = youtube.search().list(
        q=channel_name,
        part="snippet",
        type="channel",
        maxResults=1
    )
    response = request.execute()
    return response["items"][0]["snippet"]["channelId"]


In [5]:
def get_channel_stats(channel_id):
    request = youtube.channels().list(
        part="statistics",
        id=channel_id
    )
    response = request.execute()
    stats = response["items"][0]["statistics"]
    return int(stats["subscriberCount"]), int(stats["viewCount"])


In [6]:
def get_recent_videos(channel_id, max_results=10):
    request = youtube.search().list(
        channelId=channel_id,
        part="snippet",
        type="video",
        order="date",
        maxResults=max_results
    )
    response = request.execute()
    videos = []
    for item in response["items"]:
        videos.append({
            "videoId": item["id"]["videoId"],
            "title": item["snippet"]["title"],
            "publishedAt": item["snippet"]["publishedAt"]
        })
    return videos


In [7]:
def get_video_engagement(videos):
    likes, comments = [], []
    video_ids = [v["videoId"] for v in videos]

    for vid in video_ids:
        request = youtube.videos().list(
            part="statistics",
            id=vid
        )
        response = request.execute()
        stats = response["items"][0]["statistics"]

        likes.append(int(stats.get("likeCount", 0)))
        comments.append(int(stats.get("commentCount", 0)))

    avg_likes = sum(likes) / len(likes) if likes else 0
    avg_comments = sum(comments) / len(comments) if comments else 0
    
    # Get video titles
    titles = [v["title"] for v in videos]
    
    # Calculate upload frequency (videos per week)
    from datetime import datetime
    if len(videos) > 1:
        try:
            dates = []
            for v in videos:
                date_str = v["publishedAt"]
                # Handle ISO format with Z timezone
                if date_str.endswith("Z"):
                    date_str = date_str[:-1] + "+00:00"
                dates.append(datetime.fromisoformat(date_str))
            dates.sort(reverse=True)  # Most recent first
            if len(dates) > 1:
                time_span_days = (dates[0] - dates[-1]).total_seconds() / (60 * 60 * 24)  # days
                if time_span_days > 0:
                    upload_frequency = (len(videos) - 1) / (time_span_days / 7)  # videos per week
                else:
                    upload_frequency = 0
            else:
                upload_frequency = 0
        except Exception:
            upload_frequency = 0
    else:
        upload_frequency = 0
    
    return avg_likes, avg_comments, titles, upload_frequency


In [8]:
df = pd.read_csv("youtube_channels_sample.csv")
df


,channel_url
0,https://www.youtube.com/@PhysicsWallah
1,https://www.youtube.com/@PW-Foundation
2,https://www.youtube.com/@PW-JEEWallah
3,https://www.youtube.com/@PW-NEETWallah
4,https://www.youtube.com/@pwmeded
5,https://www.youtube.com/@englishpw
6,https://www.youtube.com/@AlakhSir-Class9.10
7,https://www.youtube.com/@vidyapeethpw
8,https://www.youtube.com/@PWLittleChamps


In [9]:
results = []

for url in df["channel_url"]:
    channel_name = url.split("@")[-1]

    channel_id = get_channel_id(channel_name)
    subscribers, total_views = get_channel_stats(channel_id)

    videos = get_recent_videos(channel_id)
    avg_likes, avg_comments, video_titles, upload_frequency = get_video_engagement(videos)

    engagement_rate = ((avg_likes + avg_comments) / subscribers) * 100

    results.append({
        "Channel Name": channel_name,
        "Subscribers": subscribers,
        "Total Views": total_views,
        "Avg Likes (Last 10 Videos)": int(avg_likes),
        "Avg Comments (Last 10 Videos)": int(avg_comments),
        "Engagement Rate (%)": round(engagement_rate, 3),
        "Video Titles": "; ".join(video_titles[:5]),  # Show first 5 titles
        "Upload Frequency (videos/week)": round(upload_frequency, 2)
    })

results


[{'Channel Name': 'PhysicsWallah',
  'Subscribers': 14000000,
  'Total Views': 3066040570,
  'Avg Likes (Last 10 Videos)': 519,
  'Avg Comments (Last 10 Videos)': 0,
  'Engagement Rate (%)': 0.004,
  'Video Titles': 'INDIA&#39;S BIGGEST Educational Festival🔥🔥🔥',
  'Upload Frequency (videos/week)': 0},
 {'Channel Name': 'PW-Foundation',
  'Subscribers': 6210000,
  'Total Views': 2068062124,
  'Avg Likes (Last 10 Videos)': 12342,
  'Avg Comments (Last 10 Videos)': 989,
  'Engagement Rate (%)': 0.215,
  'Video Titles': 'Complete English ( Course A &amp; Course B ) - 🚨UPDATE; Class 10 Communicative English in One Shot | Complete Course | Board Exam 2026; Class 10 ENGLISH in One Shot 🔥 Rapid Revision | Board Exam 2026; THE FINAL BATTLE 🔥 || ENGLISH Comeback in 3 Days || Anurag Sir; All The Best For Class 10th BOARDS 2026 ❤️ EXAM में Macha के आना 🔥',
  'Upload Frequency (videos/week)': 8.55},
 {'Channel Name': 'PW-JEEWallah',
  'Subscribers': 3090000,
  'Total Views': 1637525987,
  'Avg Like

In [10]:
output_df = pd.DataFrame(results)
output_df = output_df.sort_values("Engagement Rate (%)", ascending=False)
output_df["Rank"] = range(1, len(output_df) + 1)
output_df


,Channel Name,Subscribers,Total Views,Avg Likes (Last 10 Videos),Avg Comments (Last 10 Videos),Engagement Rate (%),Video Titles,Upload Frequency (videos/week),Rank
6,AlakhSir-Class9.10,1930000,326489585,72055,4268,3.955,Day 2 - complete English Comeback 🔥 #cbseclass...,7.12,1
3,PW-NEETWallah,402000,16963508,6328,613,1.727,See you in the class🤝 | @PW-NEETWallah - #umm...,1.04,2
2,PW-JEEWallah,3090000,1637525987,7273,154,0.240,FREE Physics Series😎🔥 #jeewallah #shorts #pw #...,32.08,3
1,PW-Foundation,6210000,2068062124,12342,989,0.215,Complete English ( Course A &amp; Course B ) -...,8.55,4
8,PWLittleChamps,1110000,176005088,1698,646,0.211,What a Bird Thought | Class 6th English | Comp...,0.06,5
7,vidyapeethpw,1000000,746329861,827,12,0.084,JEE Main 2026 Topper Vs Junior ⚡; BREATHING AN...,33.84,6
4,pwmeded,306000,35096831,71,1,0.024,Physiology Fact: How Nerve Signals Stop Tempor...,2.87,7
5,englishpw,680000,60802347,30,1,0.005,@PWNEETEnglish NEET Physics 150 Questions Liv...,4.74,8
0,PhysicsWallah,14000000,3066040570,519,0,0.004,INDIA&#39;S BIGGEST Educational Festival🔥🔥🔥,0.00,9


In [11]:
output_df.to_csv("youtube_analysis_output.csv", index=False)
